# Canadian Household Spending Analysis — Insurance Sector
**Analyst:** Carlos Restrepo | GLOCAL Foundation of Canada  
**Phase:** Comparative Sector Analysis — Insurance  
**Source:** Statistics Canada — SHS PUMF 2017, 2019, 2021  
**Kernel:** Python (shs2021)

## Business Objective
This notebook compares visible household insurance spending across **2017, 2019, and 2021** to identify how Canadian protection behavior evolved across property, vehicle, health/dental, disability, and mortgage-linked insurance lines.

## Core Questions
1. How did **core household insurance spending** change from 2017 to 2021?
2. Which insurance lines drove the strongest change in the period?
3. How did insurance behavior evolve by **income quintile, tenure, and province**?
4. Which patterns should be read as structural, and which may reflect the **COVID-era distortion** present in 2021?

## Method Note
This notebook focuses on a **protection-led insurance view** using line items visible in the SHS hierarchy. It is not a full insurer market model. The analysis centers on:
- `HC061` — Private health and dental plan premiums
- `HC025` — Accident or disability insurance premiums
- `SH015` — Homeowners' property insurance
- `SH019` — Mortgage-related insurance premiums
- `SH044` — Property insurance for owned secondary residences
- `TR085` — Public and private vehicle insurance premiums

Two adjacent context variables are tracked separately:
- `EP011` — Personal insurance premiums and retirement/pension fund contributions
- `ME039` — Financial services

## Interpretation Note
- **2017** and **2019** are the best pre-COVID reference years.
- **2021** should be interpreted carefully because household behavior was still influenced by COVID-era conditions, especially around mobility, home-centered risk exposure, and health-protection awareness.


In [ ]:

# ============================================================
# Cell 2 - Imports and helper functions
# ============================================================

import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 180)

base = Path(r"C:\Users\LENOVO\canadian_household_spending")
out_dir = base / 'outputs' / 'comparative' / 'insurance_sector_comparative'
os.makedirs(out_dir, exist_ok=True)


def weighted_mean(df, var, weight='WeightD'):
    if var not in df.columns or weight not in df.columns:
        return np.nan
    valid = df[[var, weight]].dropna()
    if valid.empty or valid[weight].sum() == 0:
        return np.nan
    return (valid[var] * valid[weight]).sum() / valid[weight].sum()


def weighted_positive_rate(df, var, weight='WeightD'):
    if var not in df.columns or weight not in df.columns:
        return np.nan
    valid = df[[var, weight]].dropna()
    if valid.empty or valid[weight].sum() == 0:
        return np.nan
    return (valid.loc[valid[var] > 0, weight].sum() / valid[weight].sum()) * 100


def standardize_income_column(df):
    if 'HH_TotInc' in df.columns and 'HH_TOTAL_INCOME' not in df.columns:
        return df.rename(columns={'HH_TotInc': 'HH_TOTAL_INCOME'})
    if 'HHTOTINC' in df.columns and 'HH_TOTAL_INCOME' not in df.columns:
        return df.rename(columns={'HHTOTINC': 'HH_TOTAL_INCOME'})
    return df


def ensure_column(df, candidates, target_name):
    for c in candidates:
        if c in df.columns:
            if c != target_name:
                df = df.rename(columns={c: target_name})
            return df
    df[target_name] = np.nan
    return df


def build_total(df, cols, total_name):
    existing = [c for c in cols if c in df.columns]
    if not existing:
        df[total_name] = np.nan
    else:
        df[total_name] = df[existing].fillna(0).sum(axis=1)
    return df


def load_year(year):
    path = base / 'outputs' / str(year) / f'shs_{year}_clean.csv'
    df = pd.read_csv(path)
    df = standardize_income_column(df)
    df = ensure_column(df, ['WeightD', 'WEIGHTD'], 'WeightD')
    df = ensure_column(df, ['PROV_NAME'], 'PROV_NAME')
    df = ensure_column(df, ['INCOME_QUINTILE'], 'INCOME_QUINTILE')
    df = ensure_column(df, ['TENURE_NAME'], 'TENURE_NAME')
    df['Year'] = year
    return df


insurance_map = {
    'Private health & dental': 'HC061',
    'Accident / disability': 'HC025',
    'Homeowners property': 'SH015',
    'Mortgage-related insurance': 'SH019',
    'Secondary residence property': 'SH044',
    'Vehicle insurance': 'TR085',
}

adjacent_map = {
    'Personal insurance + retirement': 'EP011',
    'Financial services': 'ME039',
}


## Load and harmonize yearly datasets
This section loads the cleaned annual datasets, standardizes common fields, and builds a comparable **core insurance** measure for all three years.

In [ ]:

# ============================================================
# Cell 3 - Load yearly datasets and build comparable totals
# ============================================================

dfs = {}
for year in [2017, 2019, 2021]:
    df = load_year(year)
    df = build_total(df, list(insurance_map.values()), 'INS_CORE_TOTAL')
    dfs[year] = df

for year, df in dfs.items():
    print(f"{year}: rows={df.shape[0]:,}, cols={df.shape[1]:,}")

harmonization = []
for year, df in dfs.items():
    row = {'Year': year}
    for label, code in insurance_map.items():
        row[f'{label} ({code})'] = code in df.columns
    for label, code in adjacent_map.items():
        row[f'{label} ({code})'] = code in df.columns
    harmonization.append(row)

harmonization_df = pd.DataFrame(harmonization)
display(harmonization_df.style.hide(axis='index'))


## National trend overview
This section builds the main national comparison: overall insurance spend, burden, participation, and adjacent long-term protection context.

In [ ]:

# ============================================================
# Cell 4 - National comparison table
# ============================================================

national_rows = []
for year, df in dfs.items():
    income_avg = weighted_mean(df, 'HH_TOTAL_INCOME')
    core_avg = weighted_mean(df, 'INS_CORE_TOTAL')
    row = {
        'Year': year,
        'Avg Household Income': income_avg,
        'Avg Core Insurance Spend': core_avg,
        'Insurance Burden %': (core_avg / income_avg * 100) if pd.notna(income_avg) and income_avg != 0 else np.nan,
        'Any Core Insurance %': weighted_positive_rate(df, 'INS_CORE_TOTAL'),
        'Avg Personal Insurance + Retirement': weighted_mean(df, 'EP011'),
        'Avg Financial Services Spend': weighted_mean(df, 'ME039'),
    }
    for label, code in insurance_map.items():
        row[label] = weighted_mean(df, code)
    national_rows.append(row)

national_df = pd.DataFrame(national_rows).round(1)
display(
    national_df.style.hide(axis='index').format({
        'Avg Household Income': '${:,.1f}',
        'Avg Core Insurance Spend': '${:,.1f}',
        'Insurance Burden %': '{:.2f}%',
        'Any Core Insurance %': '{:.1f}%',
        'Avg Personal Insurance + Retirement': '${:,.1f}',
        'Avg Financial Services Spend': '${:,.1f}',
        'Private health & dental': '${:,.1f}',
        'Accident / disability': '${:,.1f}',
        'Homeowners property': '${:,.1f}',
        'Mortgage-related insurance': '${:,.1f}',
        'Secondary residence property': '${:,.1f}',
        'Vehicle insurance': '${:,.1f}',
    })
)


In [ ]:

# ============================================================
# Cell 5 - Charts: national trends and line trajectories
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(national_df['Year'], national_df['Avg Core Insurance Spend'], marker='o', label='Core insurance')
axes[0].plot(national_df['Year'], national_df['Avg Personal Insurance + Retirement'], marker='o', label='EP011')
axes[0].plot(national_df['Year'], national_df['Avg Financial Services Spend'], marker='o', label='Financial services')
axes[0].set_title('Visible Financial Protection Spend', fontweight='bold')
axes[0].set_ylabel('CAD ($)')
axes[0].set_xlabel('Year')
axes[0].legend()

axes[1].plot(national_df['Year'], national_df['Insurance Burden %'], marker='o', label='Insurance burden %')
axes[1].plot(national_df['Year'], national_df['Any Core Insurance %'], marker='o', label='Any core insurance %')
axes[1].set_title('Burden and Participation Trend', fontweight='bold')
axes[1].set_ylabel('Percent')
axes[1].set_xlabel('Year')
axes[1].legend()

for col in ['Vehicle insurance', 'Private health & dental', 'Homeowners property', 'Accident / disability']:
    axes[2].plot(national_df['Year'], national_df[col], marker='o', label=col)
axes[2].set_title('Largest Core Insurance Lines', fontweight='bold')
axes[2].set_ylabel('CAD ($)')
axes[2].set_xlabel('Year')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig(out_dir / 'insurance_national_trends.png', dpi=150, bbox_inches='tight')
plt.show()
print('OK Chart saved!')


## Core line mix and mix shift
This section isolates the internal mix of core insurance and shows which lines gained or lost share across the period.

In [ ]:

# ============================================================
# Cell 6 - Core insurance mix and shares by year
# ============================================================

mix_rows = []
for year, df in dfs.items():
    core_total = weighted_mean(df, 'INS_CORE_TOTAL')
    for label, code in insurance_map.items():
        avg_spend = weighted_mean(df, code)
        mix_rows.append({
            'Year': year,
            'Insurance Line': label,
            'Avg Spend': avg_spend,
            'Share of Core Insurance %': (avg_spend / core_total * 100) if pd.notna(core_total) and core_total != 0 else np.nan,
            'Households with Spend %': weighted_positive_rate(df, code),
        })

mix_df = pd.DataFrame(mix_rows).round(1)
display(
    mix_df.style.hide(axis='index').format({
        'Avg Spend': '${:,.1f}',
        'Share of Core Insurance %': '{:.1f}%',
        'Households with Spend %': '{:.1f}%',
    })
)


In [ ]:

# ============================================================
# Cell 7 - Charts: mix shift by line
# ============================================================

pivot_spend = mix_df.pivot(index='Year', columns='Insurance Line', values='Avg Spend')
pivot_share = mix_df.pivot(index='Year', columns='Insurance Line', values='Share of Core Insurance %')

fig, axes = plt.subplots(1, 2, figsize=(17, 6))

for col in pivot_spend.columns:
    axes[0].plot(pivot_spend.index, pivot_spend[col], marker='o', label=col)
axes[0].set_title('Insurance Line Spending Trends', fontweight='bold')
axes[0].set_ylabel('CAD ($)')
axes[0].set_xlabel('Year')
axes[0].legend(fontsize=8)

for col in pivot_share.columns:
    axes[1].plot(pivot_share.index, pivot_share[col], marker='o', label=col)
axes[1].set_title('Insurance Line Share of Core Insurance', fontweight='bold')
axes[1].set_ylabel('Percent of core insurance')
axes[1].set_xlabel('Year')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(out_dir / 'insurance_line_trends.png', dpi=150, bbox_inches='tight')
plt.show()
print('OK Chart saved!')


## Comparative view by income quintile
Income is the key segmentation lens for insurance because it affects both the ability to buy protection and the relative burden of that protection.

In [ ]:

# ============================================================
# Cell 8 - Quintile comparison across years
# ============================================================

quintiles = ['Q1 - Lowest', 'Q2', 'Q3', 'Q4', 'Q5 - Highest']
q_rows = []
for year, df in dfs.items():
    for q in quintiles:
        subset = df[df['INCOME_QUINTILE'] == q].copy()
        income_avg = weighted_mean(subset, 'HH_TOTAL_INCOME')
        core_avg = weighted_mean(subset, 'INS_CORE_TOTAL')
        row = {
            'Year': year,
            'Quintile': q,
            'Avg Income': income_avg,
            'Core Insurance': core_avg,
            'Insurance Burden %': (core_avg / income_avg * 100) if pd.notna(income_avg) and income_avg != 0 else np.nan,
            'Any Core Insurance %': weighted_positive_rate(subset, 'INS_CORE_TOTAL'),
            'Vehicle insurance': weighted_mean(subset, 'TR085'),
            'Homeowners property': weighted_mean(subset, 'SH015'),
            'Private health & dental': weighted_mean(subset, 'HC061'),
            'Accident / disability': weighted_mean(subset, 'HC025'),
            'Mortgage-related insurance': weighted_mean(subset, 'SH019'),
            'Personal insurance + retirement': weighted_mean(subset, 'EP011'),
        }
        q_rows.append(row)

quintile_df = pd.DataFrame(q_rows).round(1)
display(
    quintile_df.style.hide(axis='index').format({
        'Avg Income': '${:,.1f}',
        'Core Insurance': '${:,.1f}',
        'Insurance Burden %': '{:.2f}%',
        'Any Core Insurance %': '{:.1f}%',
        'Vehicle insurance': '${:,.1f}',
        'Homeowners property': '${:,.1f}',
        'Private health & dental': '${:,.1f}',
        'Accident / disability': '${:,.1f}',
        'Mortgage-related insurance': '${:,.1f}',
        'Personal insurance + retirement': '${:,.1f}',
    })
)


In [ ]:

# ============================================================
# Cell 9 - Charts: quintile trends
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(17, 6))

for q in quintiles:
    s = quintile_df[quintile_df['Quintile'] == q]
    axes[0].plot(s['Year'], s['Core Insurance'], marker='o', label=q)
axes[0].set_title('Core Insurance Spend by Income Quintile', fontweight='bold')
axes[0].set_ylabel('CAD ($)')
axes[0].set_xlabel('Year')
axes[0].legend(fontsize=8)

for q in quintiles:
    s = quintile_df[quintile_df['Quintile'] == q]
    axes[1].plot(s['Year'], s['Insurance Burden %'], marker='o', label=q)
axes[1].set_title('Insurance Burden by Income Quintile', fontweight='bold')
axes[1].set_ylabel('% of household income')
axes[1].set_xlabel('Year')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(out_dir / 'insurance_quintile_trends.png', dpi=150, bbox_inches='tight')
plt.show()
print('OK Chart saved!')


## Tenure matters for insurance structure
Tenure is a critical lens because property, mortgage-linked insurance, and renter exposure produce very different protection baskets.

In [ ]:

# ============================================================
# Cell 10 - Tenure comparison across years
# ============================================================

tenure_order = ['Owner with mortgage', 'Owner without mortgage', 'Renter']
tenure_rows = []
for year, df in dfs.items():
    for tenure in tenure_order:
        subset = df[df['TENURE_NAME'] == tenure].copy()
        income_avg = weighted_mean(subset, 'HH_TOTAL_INCOME')
        core_avg = weighted_mean(subset, 'INS_CORE_TOTAL')
        row = {
            'Year': year,
            'Tenure': tenure,
            'Avg Income': income_avg,
            'Core Insurance': core_avg,
            'Insurance Burden %': (core_avg / income_avg * 100) if pd.notna(income_avg) and income_avg != 0 else np.nan,
            'Any Core Insurance %': weighted_positive_rate(subset, 'INS_CORE_TOTAL'),
            'Vehicle insurance': weighted_mean(subset, 'TR085'),
            'Homeowners property': weighted_mean(subset, 'SH015'),
            'Private health & dental': weighted_mean(subset, 'HC061'),
            'Accident / disability': weighted_mean(subset, 'HC025'),
            'Mortgage-related insurance': weighted_mean(subset, 'SH019'),
        }
        tenure_rows.append(row)

tenure_df = pd.DataFrame(tenure_rows).round(1)
display(
    tenure_df.style.hide(axis='index').format({
        'Avg Income': '${:,.1f}',
        'Core Insurance': '${:,.1f}',
        'Insurance Burden %': '{:.2f}%',
        'Any Core Insurance %': '{:.1f}%',
        'Vehicle insurance': '${:,.1f}',
        'Homeowners property': '${:,.1f}',
        'Private health & dental': '${:,.1f}',
        'Accident / disability': '${:,.1f}',
        'Mortgage-related insurance': '${:,.1f}',
    })
)


In [ ]:

# ============================================================
# Cell 11 - Charts: tenure trend comparison
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(17, 6))

for tenure in tenure_order:
    s = tenure_df[tenure_df['Tenure'] == tenure]
    axes[0].plot(s['Year'], s['Core Insurance'], marker='o', label=tenure)
axes[0].set_title('Core Insurance Spend by Tenure', fontweight='bold')
axes[0].set_ylabel('CAD ($)')
axes[0].set_xlabel('Year')
axes[0].legend(fontsize=8)

for tenure in tenure_order:
    s = tenure_df[tenure_df['Tenure'] == tenure]
    axes[1].plot(s['Year'], s['Insurance Burden %'], marker='o', label=tenure)
axes[1].set_title('Insurance Burden by Tenure', fontweight='bold')
axes[1].set_ylabel('% of household income')
axes[1].set_xlabel('Year')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(out_dir / 'insurance_tenure_trends.png', dpi=150, bbox_inches='tight')
plt.show()
print('OK Chart saved!')


## Provincial comparison
The goal here is not to overfit small differences, but to identify which provinces consistently look like heavier, lighter, or differently structured insurance markets.

In [ ]:

# ============================================================
# Cell 12 - Provincial comparison and change vs 2017
# ============================================================

prov_rows = []
for year, df in dfs.items():
    for prov in sorted(df['PROV_NAME'].dropna().unique()):
        subset = df[df['PROV_NAME'] == prov].copy()
        income_avg = weighted_mean(subset, 'HH_TOTAL_INCOME')
        core_avg = weighted_mean(subset, 'INS_CORE_TOTAL')
        prov_rows.append({
            'Year': year,
            'Province': prov,
            'Avg Income': income_avg,
            'Core Insurance': core_avg,
            'Insurance Burden %': (core_avg / income_avg * 100) if pd.notna(income_avg) and income_avg != 0 else np.nan,
            'Vehicle insurance': weighted_mean(subset, 'TR085'),
            'Homeowners property': weighted_mean(subset, 'SH015'),
            'Private health & dental': weighted_mean(subset, 'HC061'),
        })

prov_df = pd.DataFrame(prov_rows).round(1)

prov_2021 = prov_df[prov_df['Year'] == 2021].sort_values('Core Insurance', ascending=False)
display(
    prov_2021.style.hide(axis='index').format({
        'Avg Income': '${:,.1f}',
        'Core Insurance': '${:,.1f}',
        'Insurance Burden %': '{:.2f}%',
        'Vehicle insurance': '${:,.1f}',
        'Homeowners property': '${:,.1f}',
        'Private health & dental': '${:,.1f}',
    })
)


In [ ]:

# ============================================================
# Cell 13 - Charts: provincial leaders and change over time
# ============================================================

prov_change = prov_df.pivot(index='Province', columns='Year', values='Core Insurance')
prov_change['2017_to_2021_Change'] = prov_change[2021] - prov_change[2017]
prov_change = prov_change.sort_values('2017_to_2021_Change', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

axes[0].barh(prov_change.index, prov_change['2017_to_2021_Change'])
axes[0].set_title('Change in Core Insurance Spend: 2017 to 2021', fontweight='bold')
axes[0].set_xlabel('CAD ($)')

top_2021 = prov_df[prov_df['Year'] == 2021].sort_values('Insurance Burden %', ascending=False)
axes[1].barh(top_2021['Province'], top_2021['Insurance Burden %'])
axes[1].set_title('Insurance Burden by Province (2021)', fontweight='bold')
axes[1].set_xlabel('% of household income')

plt.tight_layout()
plt.savefig(out_dir / 'insurance_province_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('OK Chart saved!')


## Protection profile scorecard
This section condenses the three-year comparison into a small scorecard that can be used for executive interpretation.

In [ ]:

# ============================================================
# Cell 14 - Comparative executive scorecard
# ============================================================

mix_leaders = mix_df.sort_values(['Year', 'Avg Spend'], ascending=[True, False]).groupby('Year').first().reset_index()

scorecard_rows = []
for year in [2017, 2019, 2021]:
    nat = national_df[national_df['Year'] == year].iloc[0]
    q1 = quintile_df[(quintile_df['Year'] == year) & (quintile_df['Quintile'] == 'Q1 - Lowest')].iloc[0]
    q5 = quintile_df[(quintile_df['Year'] == year) & (quintile_df['Quintile'] == 'Q5 - Highest')].iloc[0]
    leader = mix_leaders[mix_leaders['Year'] == year].iloc[0]
    own_m = tenure_df[(tenure_df['Year'] == year) & (tenure_df['Tenure'] == 'Owner with mortgage')].iloc[0]
    renter = tenure_df[(tenure_df['Year'] == year) & (tenure_df['Tenure'] == 'Renter')].iloc[0]
    scorecard_rows.append({
        'Year': year,
        'Avg Core Insurance Spend': nat['Avg Core Insurance Spend'],
        'Insurance Burden %': nat['Insurance Burden %'],
        'Participation %': nat['Any Core Insurance %'],
        'Top Insurance Line': leader['Insurance Line'],
        'Top Line Avg Spend': leader['Avg Spend'],
        'Q1 Burden %': q1['Insurance Burden %'],
        'Q5 Burden %': q5['Insurance Burden %'],
        'Owner w/ Mortgage Core Insurance': own_m['Core Insurance'],
        'Renter Core Insurance': renter['Core Insurance'],
    })

scorecard_df = pd.DataFrame(scorecard_rows).round(1)
display(
    scorecard_df.style.hide(axis='index').format({
        'Avg Core Insurance Spend': '${:,.1f}',
        'Insurance Burden %': '{:.2f}%',
        'Participation %': '{:.1f}%',
        'Top Line Avg Spend': '${:,.1f}',
        'Q1 Burden %': '{:.2f}%',
        'Q5 Burden %': '{:.2f}%',
        'Owner w/ Mortgage Core Insurance': '${:,.1f}',
        'Renter Core Insurance': '${:,.1f}',
    })
)


## Comparative strategic interpretation
- The most important structural distinction in this dataset is between **core protection** and **broader long-term financial protection context**.
- **Income** should be treated as the main segmentation variable: lower-income households are more exposed to protection burden, while higher-income households carry broader and deeper insurance baskets.
- **Tenure** is essential because owners with mortgages tend to combine vehicle, home, and mortgage-linked protection in a way renters do not.
- **Provincial insurance markets** should be interpreted as distinct risk and protection ecosystems, not as one homogeneous national market.
- **2021 should not be treated as a fully normal year**. Any increase or mix shift observed in that year may reflect both structural insurance behavior and COVID-era distortion in mobility, home exposure, and health-protection awareness.
- For strategy, the strongest lenses are:
  - **core protection affordability**
  - **mortgage-linked protection intensity**
  - **vehicle/property mix**
  - **health/dental participation**
  - **long-term protection orientation via EP011**
